## **Imports**

In [ ]:
#!pip install -U transformers

## **Hugging Face Token**

In [1]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("my_hf_token")

In [2]:
from huggingface_hub import login
login(token=hf_token)

## **Imports**

In [3]:
import json
import re
import math
import random
import copy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
import gc, traceback

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [4]:
MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
HF_TOKEN = hf_token

## **Load Model**

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    dtype=torch.float16,      # NOT torch_dtype (deprecated, as you already found)
    device_map="auto",
)
model.eval()

DEVICE = next(model.parameters()).device
NUM_LAYERS = model.config.num_hidden_layers
HIDDEN_DIM = model.config.hidden_size
print(f"Loaded {MODEL_NAME}: {NUM_LAYERS} layers, hidden_dim={HIDDEN_DIM}, device={DEVICE}")

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Loaded meta-llama/Llama-3.2-3B-Instruct: 28 layers, hidden_dim=3072, device=cuda:0


In [6]:
# freeze model params

for p in model.parameters():
    p.requires_grad_(False)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params in base model (should be 0): {trainable}")

Trainable params in base model (should be 0): 0


## **Dataset Loading**

In [7]:
df = pd.read_csv("/kaggle/input/datasets/anarvaaa/dataset-for-ssv/combined_pico_questions.csv")  # update path if running locally
# If running outside Kaggle, point this at your local copy instead
print(df.shape)
df.head(3)


(100, 7)


,sr.no.,speciality,question,P,I,C,O
0,1,Cardiology,In adults with heart failure with preserved ej...,Adults with HFpEF (LVEF ≥ 50%),"SGLT2 inhibitors (e.g., empagliflozin, dapagli...",Placebo + standard care,Composite of cardiovascular death and hospital...
1,2,Cardiology,In patients undergoing percutaneous coronary i...,Adults undergoing PCI for acute coronary syndrome,Short-term DAPT (1–3 mo) followed by P2Y12 inh...,Standard DAPT for 12 months,Bleeding Academic Research Consortium (BARC) t...
2,3,Cardiology,In adults with severe asymptomatic aortic sten...,Adults with severe asymptomatic aortic stenosis,Early elective AVR (surgical or transcatheter),Conservative watchful waiting,All-cause mortality; myocardial infarction at ...


In [8]:
def format_target(row):
    return (
        f"P: {row['P']}\n"
        f"I: {row['I']}\n"
        f"C: {row['C']}\n"
        f"O: {row['O']}"
    )

'''
def format_prompt(row):
    # Deliberately minimal / un-instructed prompt. We are NOT telling the model
    # to produce PICO format via prompting -- the steering vector is what should
    # push the model toward that structure.
    return f"{row['research_question'].strip()}\n"
'''
# new prompt having instruction too for training
def format_prompt(row):
    return (
        "Extract the PICO elements from the following research question.\n\n"
        "P (Population): The group of patients or participants being studied.\n"
        "I (Intervention/Exposure): The treatment, intervention, or exposure being evaluated.\n"
        "C (Comparison): The comparator, control, or alternative intervention.\n"
        "O (Outcome): The outcome being measured or predicted.\n\n"
        "Return the extracted elements in exactly this format:\n"
        "P: ...\n"
        "I: ...\n"
        "C: ...\n"
        "O: ...\n\n"
        f"Research question:\n{row['question'].strip()}\n"
    )

examples = []
for _, row in df.iterrows():
    examples.append({
        "prompt": format_prompt(row),
        "target": format_target(row) + tokenizer.eos_token,
    })

random.shuffle(examples)
n_val = max(5, int(0.1 * len(examples)))
val_examples = examples[:n_val]
train_examples = examples[n_val:]
print(f"train={len(train_examples)}  val={len(val_examples)}")
print("--- sample ---")
print(train_examples[0]["prompt"])
print(train_examples[0]["target"])

train=90  val=10
--- sample ---
Extract the PICO elements from the following research question.

P (Population): The group of patients or participants being studied.
I (Intervention/Exposure): The treatment, intervention, or exposure being evaluated.
C (Comparison): The comparator, control, or alternative intervention.
O (Outcome): The outcome being measured or predicted.

Return the extracted elements in exactly this format:
P: ...
I: ...
C: ...
O: ...

Research question:
(Diagnostic) In patients with suspected Sjögren’s disease, does salivary gland ultrasound compared with lip biopsy accurately diagnose SjD?

P: Adults with sicca symptoms, anti-Ro negative
I: Major salivary gland ultrasound (SGUS)
C: Labial salivary gland biopsy (focus score \geq1)
O: Sensitivity and specificity for SjD classification<|eot_id|>


## **S-SPR**

In [9]:
class SPSR(nn.Module):
    """Single-layer Prompt Steering Replacement module (Eq. 7-8)."""
    def __init__(self, dim, dtype=torch.float32):
        super().__init__()
        self.z = nn.Parameter(torch.randn(dim, dtype=dtype) * 0.01)   # steering vector z_attr,l
        self.w = nn.Parameter(torch.randn(dim, dtype=dtype) * 0.01)   # probe weight w_attr,l
        self.b = nn.Parameter(torch.zeros(1, dtype=dtype) * 0.5)            # probe bias b_attr,l

    def coefficient(self, h):
        """lambda(h) = ReLU(h . w + b), computed per token. h: [..., dim] -> [..., 1]"""
        w = self.w.to(device=h.device, dtype=h.dtype)
        b = self.b.to(device=h.device, dtype=h.dtype)
        pre = torch.matmul(h, w) + b
        lam = F.leaky_relu(pre, negative_slope=0.01) if self.training else F.relu(pre)
        return lam.unsqueeze(-1)

    def forward(self, h, alpha):
        z = self.z.to(device=h.device, dtype=h.dtype)
        lam = self.coefficient(h)          # [..., 1], per-token scalar
        steered = h + alpha * lam * z
        return steered, lam

class SteeringHook:
    """Forward hook that injects an SPSR intervention into a decoder layer's output.

    `enabled=False` acts as a no-op passthrough, so the exact same model object
    can be used for steered vs. unsteered ("prompting-only") generation.
    Stores the last computed per-token lambda (`last_lambda`) so the training
    loop can apply the Sec. 3.5 regularization term.
    """
    def __init__(self, spsr: SPSR):
        self.sv = spsr
        self.enabled = False
        self.alpha = 0.0
        self.handle = None
        self.last_lambda = None

    def __call__(self, module, inputs, output):
        if not self.enabled:
            return output
        if isinstance(output, tuple):
            hidden_states = output[0]
            hidden_states, lam = self.sv(hidden_states, self.alpha)
            self.last_lambda = lam
            return (hidden_states,) + output[1:]
        else:
            hidden_states, lam = self.sv(output, self.alpha)
            self.last_lambda = lam
            return hidden_states

    def attach(self, layer_module):
        self.handle = layer_module.register_forward_hook(self)
        return self.handle

    def remove(self):
        if self.handle is not None:
            self.handle.remove()
            self.handle = None

In [10]:
# Pick an intervention layer. Middle-ish layers tend to encode more abstract /
# format-level concepts (see Fig. 13 in the paper -- steerability varies by
# layer). Gemma-2B has NUM_LAYERS layers; we default to roughly the 40% mark.
# but for llama model, we implement layer 16 as recommended in table 4, appendix C
STEER_LAYER = 16
assert STEER_LAYER < NUM_LAYERS, f"Llama-3.2-3B-Instruct only has {NUM_LAYERS} layers"

# With device_map="auto" the model can be sharded across multiple GPUs, so the
# layer we hook may not live on the same device as model.parameters()[0]
# (which is what DEVICE points to). Find the *actual* device of the hooked
# layer and put the S-PSR module there -- the hook itself also casts on
# every call as a safety net, but starting on the right device avoids
# unnecessary cross-GPU copies.
layer_device = next(model.model.layers[STEER_LAYER].parameters()).device
print(f"Hooking layer index {STEER_LAYER} / {NUM_LAYERS} (on {layer_device})")

# NOTE: keep S-PSR params in float32 even though the model runs in float16.
# This is the standard "fp32 master weights" pattern for mixed precision:
# AdamW's default eps=1e-8 underflows to 0 once cast into fp16 (fp16's
# smallest normal value is ~6e-5), which silently produces NaNs via
# division-by-~0 in the Adam update. fp32 params + fp32 optimizer state
# avoids this; SPSR.forward already casts down to h.dtype only for the
# actual addition into the model's fp16 hidden states.
psr = SPSR(HIDDEN_DIM, dtype=torch.float32).to(layer_device)
steering_hook = SteeringHook(psr)
steering_hook.attach(model.model.layers[STEER_LAYER])

n_trainable = sum(p.numel() for p in psr.parameters() if p.requires_grad)
print(f"Trainable S-PSR params: {n_trainable}  (z, w: {HIDDEN_DIM} each, b: 1)")

Hooking layer index 16 / 28 (on cuda:1)
Trainable S-PSR params: 6145  (z, w: 3072 each, b: 1)


## **Batch Construction**

In [11]:
def build_batch(example, device=DEVICE):
    prompt_ids = tokenizer(example["prompt"], return_tensors="pt", add_special_tokens=True).input_ids[0]
    target_ids = tokenizer(example["target"], return_tensors="pt", add_special_tokens=False).input_ids[0]

    input_ids = torch.cat([prompt_ids, target_ids], dim=0)
    labels = input_ids.clone()
    labels[: len(prompt_ids)] = -100  # only train on the PICO-formatted continuation

    input_ids = input_ids.unsqueeze(0).to(device)
    labels = labels.unsqueeze(0).to(device)
    attention_mask = torch.ones_like(input_ids)
    return input_ids, attention_mask, labels

## **Training**

In [12]:
TRAIN_FACTORS = [2.0, 4.0, 6.0, 8.0, 10.0, 12.0, 14.0, 16.0]
LR = 1e-3
EPOCHS = 10
GRAD_CLIP = 1.0
LAMBDA_REG = 0.01   # weight of the Sec. 3.5 regularization term; set to 0.0 to disable

optimizer = torch.optim.AdamW(psr.parameters(), lr=LR, weight_decay=0.0)

def run_forward(input_ids, attention_mask, labels, alpha, lambda_reg=LAMBDA_REG):
    steering_hook.enabled = True
    steering_hook.alpha = alpha
    out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    loss = out.loss

    if lambda_reg > 0:
        lam = steering_hook.last_lambda  # [batch, seq, 1], computed on this forward pass
        resp_mask = (labels != -100).to(device=lam.device, dtype=lam.dtype).unsqueeze(-1)  # only response tokens
        lam_sum = (lam * resp_mask).sum()
        reg = torch.clamp(1.0 - lam_sum, min=0.0)
        loss = loss + lambda_reg * reg.to(loss.device)

    return loss

@torch.no_grad()
def evaluate(examples, alpha):
    steering_hook.enabled = True
    steering_hook.alpha = alpha
    losses = []
    for ex in examples:
        input_ids, attention_mask, labels = build_batch(ex)
        out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        losses.append(out.loss.item())
    return float(np.mean(losses))

In [13]:
history = {"train_loss": [], "val_loss": []}
skipped_steps = 0

best_val_loss = float("inf")
best_psr_state = None
best_epoch = None

for epoch in range(EPOCHS):
    psr.train()
    random.shuffle(train_examples)
    epoch_losses = []

    for ex in train_examples:
        input_ids, attention_mask, labels = build_batch(ex)
        alpha = random.choice(TRAIN_FACTORS)

        optimizer.zero_grad()
        loss = run_forward(input_ids, attention_mask, labels, alpha)

        if not torch.isfinite(loss):
            skipped_steps += 1
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(psr.parameters(), GRAD_CLIP)
        optimizer.step()

        epoch_losses.append(loss.item())

    train_loss = float(np.mean(epoch_losses)) if epoch_losses else float("nan")

    # Validate at the middle of the training factor range
    psr.eval()
    val_loss = evaluate(
        val_examples,
        alpha=TRAIN_FACTORS[len(TRAIN_FACTORS) // 2]
    )

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    # Save best validation checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        best_psr_state = {
            k: v.detach().cpu().clone()
            for k, v in psr.state_dict().items()
        }

    print(
        f"epoch {epoch+1:02d}/{EPOCHS}  "
        f"train_loss={train_loss:.4f}  "
        f"val_loss={val_loss:.4f}  "
        f"(best={best_val_loss:.4f} @ epoch {best_epoch}, "
        f"skipped {skipped_steps} bad steps so far)"
    )

# Restore the best validation checkpoint
psr.load_state_dict(best_psr_state)

print(f"\nRestored best S-PSR checkpoint: epoch {best_epoch}")
print(f"Best validation loss: {best_val_loss:.4f}")

epoch 01/10  train_loss=1.4867  val_loss=1.1310  (best=1.1310 @ epoch 1, skipped 0 bad steps so far)
epoch 02/10  train_loss=1.2780  val_loss=1.1175  (best=1.1175 @ epoch 2, skipped 0 bad steps so far)
epoch 03/10  train_loss=1.2193  val_loss=1.2192  (best=1.1175 @ epoch 2, skipped 0 bad steps so far)
epoch 04/10  train_loss=1.2212  val_loss=1.1337  (best=1.1175 @ epoch 2, skipped 0 bad steps so far)
epoch 05/10  train_loss=1.1940  val_loss=1.1624  (best=1.1175 @ epoch 2, skipped 0 bad steps so far)
epoch 06/10  train_loss=1.1752  val_loss=1.1087  (best=1.1087 @ epoch 6, skipped 0 bad steps so far)
epoch 07/10  train_loss=1.1484  val_loss=1.1286  (best=1.1087 @ epoch 6, skipped 0 bad steps so far)
epoch 08/10  train_loss=1.1432  val_loss=1.1000  (best=1.1000 @ epoch 8, skipped 0 bad steps so far)
epoch 09/10  train_loss=1.1512  val_loss=1.1059  (best=1.1000 @ epoch 8, skipped 0 bad steps so far)
epoch 10/10  train_loss=1.1294  val_loss=1.1303  (best=1.1000 @ epoch 8, skipped 0 bad step

## **Best Steering Factor**

In [14]:
candidate_factors = [1.0, 2.0, 4.0, 6.0, 8.0, 10.0, 12.0, 16.0, 20.0]
factor_scores = {}
for f in candidate_factors:
    factor_scores[f] = evaluate(val_examples, alpha=f)
    print(f"alpha={f:>5.1f}  val_loss={factor_scores[f]:.4f}")

BEST_ALPHA = min(factor_scores, key=factor_scores.get)
print(f"\nBest steering factor: {BEST_ALPHA}")

alpha=  1.0  val_loss=1.3230
alpha=  2.0  val_loss=1.1868
alpha=  4.0  val_loss=1.0845
alpha=  6.0  val_loss=1.0660
alpha=  8.0  val_loss=1.0773
alpha= 10.0  val_loss=1.1000
alpha= 12.0  val_loss=1.1286
alpha= 16.0  val_loss=1.2090
alpha= 20.0  val_loss=1.3570

Best steering factor: 6.0


In [15]:
@torch.no_grad()
def generate(prompt, alpha=None, max_new_tokens=80):
    steering_hook.enabled = alpha is not None
    steering_hook.alpha = alpha if alpha is not None else 0.0

    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=1.0,
        pad_token_id=tokenizer.pad_token_id,
    )
    text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    steering_hook.enabled = False
    return text

In [16]:
'''
for ex in val_examples[:3]:
    print("=" * 80)
    print("PROMPT:", ex["prompt"].strip())
    print("-" * 80)
    print("UNSTEERED (no hook, no PICO instruction):")
    print(generate(ex["prompt"], alpha=None))
    print("-" * 80)
    print(f"STEERED (S-PSR, alpha={BEST_ALPHA}):")
    print(generate(ex["prompt"], alpha=BEST_ALPHA))
    print("-" * 80)
    print("GROUND TRUTH:")
    print(ex["target"].replace(tokenizer.eos_token, ""))
'''

for ex in val_examples[:5]:
    print("=" * 80)
    print("PROMPT:")
    print(ex["prompt"].strip())

    print("-" * 80)
    print("UNSTEERED (instruction + no S-PSR):")
    print(generate(ex["prompt"], alpha=None))

    print("-" * 80)
    print(f"STEERED (instruction + S-PSR, alpha={BEST_ALPHA}):")
    print(generate(ex["prompt"], alpha=BEST_ALPHA))

    print("-" * 80)
    print("GROUND TRUTH:")
    print(ex["target"].replace(tokenizer.eos_token, ""))

The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


PROMPT:
Extract the PICO elements from the following research question.

P (Population): The group of patients or participants being studied.
I (Intervention/Exposure): The treatment, intervention, or exposure being evaluated.
C (Comparison): The comparator, control, or alternative intervention.
O (Outcome): The outcome being measured or predicted.

Return the extracted elements in exactly this format:
P: ...
I: ...
C: ...
O: ...

Research question:
In hospitalized adults with severe acute respiratory distress syndrome (ARDS), does early prone positioning (≥16 hours/day) compared with supine positioning reduce 28-day all-cause mortality?
--------------------------------------------------------------------------------
UNSTEERED (instruction + no S-PSR):
P: Hospitalized adults with severe ARDS
I: Early prone positioning (≥16 hours/day) vs. supine positioning
C: Supine positioning
O: 28-day all-cause mortality

The final answer is: 
P: Hospitalized adults with severe ARDS
I: Early prone p

## **Per token lambda**

In [ ]:
@torch.no_grad()
def inspect_lambda(example, alpha):
    input_ids, attention_mask, labels = build_batch(example)
    steering_hook.enabled = True
    steering_hook.alpha = alpha
    _ = model(input_ids=input_ids, attention_mask=attention_mask)
    lam = steering_hook.last_lambda[0, :, 0].detach().float().cpu().numpy()
    steering_hook.enabled = False

    tokens = tokenizer.convert_ids_to_tokens(input_ids[0].cpu().tolist())
    for tok, l in zip(tokens, lam):
        print(f"{tok!r:>20}  lambda={l:.4f}")

inspect_lambda(val_examples[0], alpha=BEST_ALPHA)

In [ ]:
torch.save(
    {
        "z": psr.z.detach().cpu(),
        "w": psr.w.detach().cpu(),
        "b": psr.b.detach().cpu(),
        "layer": STEER_LAYER,
        "best_alpha": BEST_ALPHA,
        "model_name": MODEL_NAME,
        "method": "S-PSR_LL",
    },
    "pico_spsr_vector.pt",
)
print("Saved pico_spsr_vector.pt")